# Batch DQN Trading - Improved Version

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import random
import torch
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings("ignore")

import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

import sys
sys.path.append('../')

# IMPORTANT: Use the IMPROVED environment
from src.trading_env_improved import ImprovedTradingEnv

from sklearn.preprocessing import MinMaxScaler

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import traceback

In [ ]:
# Number of seeds to run per scenario
N_SEEDS = 3 

# Specific seeds for reproducibility (document these in your paper!)
SEEDS = [42, 123, 456, 789, 1000, 2024, 3141, 5926, 8675, 3090, 7777, 9999, 1337, 4444, 5555, 6666, 8888, 1111, 2222, 3333][:N_SEEDS]

# Use deterministic mode (slower but fully reproducible)
USE_DETERMINISTIC = True

# Use single environment (required for determinism, but slower)
# Set to False for faster training but less reproducible
USE_SINGLE_ENV = True

print(f"Running {N_SEEDS} seeds per scenario: {SEEDS}")
print(f"Deterministic mode: {USE_DETERMINISTIC}")
print(f"Single environment: {USE_SINGLE_ENV}")

In [ ]:
def set_all_seeds(seed: int):
    """
    Set all random seeds for reproducibility.
    
    This function ensures that:
    - Python's random module is seeded
    - NumPy's random state is seeded
    - PyTorch's random state is seeded (CPU and GPU)
    - CUDA operations are deterministic (if enabled)
    
    Parameters
    ----------
    seed : int
        Random seed value
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    if USE_DETERMINISTIC:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed

In [ ]:
timeframe = '1d'
# timeframe = '1h'

In [ ]:
## Load scenario_config
scenario_config = pd.read_csv(f'../config_files/scenarios_config_{timeframe}_baseline_v3.csv')

# Format start_train_date, end_train_date, start_val_date, end_val_date
scenario_config['start_train_date'] = pd.to_datetime(scenario_config['start_train_date'], format='%d/%m/%Y')
scenario_config['end_train_date'] = pd.to_datetime(scenario_config['end_train_date'], format='%d/%m/%Y')
scenario_config['start_val_date'] = pd.to_datetime(scenario_config['start_val_date'], format='%d/%m/%Y')
scenario_config['end_val_date'] = pd.to_datetime(scenario_config['end_val_date'], format='%d/%m/%Y')

In [ ]:
# Load `feature_family.json` to get the feature family
with open('../config_files/feature_family_v3.json') as f:
    feature_family = json.load(f)

In [ ]:
def get_family_members(family_name, data):
    # Parse the JSON data if it's a string, otherwise use it as is
    if isinstance(data, str):
        data = json.loads(data)
    
    # Find the family with the given name
    for family in data['Families']:
        if family['Name'] == family_name:
            # Return a list of member names
            return [member['Name'] for member in family['Members']]
    
    # If family not found, return None or an empty list
    return None

In [ ]:
# For a given scenario from scenario_config, get each column into a new variable

def get_scenario_data(scenario_name, scenario_config):
    # Get the scenario row
    scenario = scenario_config[scenario_config['scenario'] == scenario_name].iloc[0]
    
    # Get the data for the scenario
    scenario_data = {
        'asset': scenario['asset'],
        'feature_family': scenario['feature_family'],
        'start_train_date': scenario['start_train_date'],
        'end_train_date': scenario['end_train_date'],
        'start_val_date': scenario['start_val_date'],
        'end_val_date': scenario['end_val_date'],
        'feature_path': scenario['feature_path'],
        'feature_file': scenario['feature_file'],
        'raw_path': scenario['raw_path'],
        'raw_file': scenario['raw_file'],
    }
    
    return scenario_data

## Configure Transaction Costs

Set realistic transaction costs based on asset type

In [ ]:
def get_transaction_cost(asset_name):
    """
    Determine transaction cost based on asset type
    
    Crypto (USDT pairs): 0.02% (0.0002) - Binance maker fee
    Traditional assets: 0.01% (0.0001)
    """
    if 'USDT' in asset_name:
        return 0.0002  # 0.02% for crypto
    else:
        return 0.0001  # 0.01% for traditional assets

# Test
print(f"BTC/USDT transaction cost: {get_transaction_cost('BTCUSDT')*100:.3f}%")
print(f"SPY transaction cost: {get_transaction_cost('SPY')*100:.3f}%")

In [ ]:
def calculate_feature_variability(series: pd.Series) -> Dict[str, float]:
    """
    Calculate multiple variability metrics for a single feature.
    
    Returns dict with:
    - std: Standard deviation
    - cv: Coefficient of variation (std/mean) - scale-independent
    - iqr: Interquartile range
    - range_ratio: (max-min)/mean - relative range
    - unique_ratio: Number of unique values / total values
    - mad: Mean absolute deviation
    - entropy: Shannon entropy of binned values
    """
    # Handle NaN/inf
    clean_series = series.replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(clean_series) < 10:
        return {
            'std': 0, 'cv': 0, 'iqr': 0, 'range_ratio': 0,
            'unique_ratio': 0, 'mad': 0, 'entropy': 0
        }
    
    mean_val = clean_series.mean()
    std_val = clean_series.std()
    
    # Coefficient of Variation (handle zero mean)
    cv = std_val / abs(mean_val) if abs(mean_val) > 1e-10 else 0
    
    # Interquartile Range
    q75, q25 = np.percentile(clean_series, [75, 25])
    iqr = q75 - q25
    
    # Range ratio
    range_val = clean_series.max() - clean_series.min()
    range_ratio = range_val / abs(mean_val) if abs(mean_val) > 1e-10 else range_val
    
    # Unique ratio
    unique_ratio = clean_series.nunique() / len(clean_series)
    
    # Mean Absolute Deviation
    mad = (clean_series - mean_val).abs().mean()
    
    # Entropy (binned)
    try:
        hist, _ = np.histogram(clean_series, bins=20)
        hist = hist / hist.sum()
        hist = hist[hist > 0]  # Remove zeros for log
        entropy = -np.sum(hist * np.log2(hist))
    except:
        entropy = 0
    
    return {
        'std': std_val,
        'cv': cv,
        'iqr': iqr,
        'range_ratio': range_ratio,
        'unique_ratio': unique_ratio,
        'mad': mad,
        'entropy': entropy
    }


def is_feature_flat(
    series: pd.Series,
    cv_threshold: float = 0.01,
    unique_threshold: float = 0.01,
    entropy_threshold: float = 1.0,
    method: str = 'combined'
) -> Tuple[bool, Dict[str, float]]:
    """
    Determine if a feature is flat/near-flat.
    
    Parameters:
    -----------
    series : pd.Series
        The feature values
    cv_threshold : float
        Coefficient of variation threshold. Below this = flat.
        Default 0.01 means std < 1% of mean
    unique_threshold : float
        Unique ratio threshold. Below this = flat.
        Default 0.01 means <1% unique values
    entropy_threshold : float
        Entropy threshold. Below this = flat.
        Default 1.0 (max entropy for 20 bins is ~4.3)
    method : str
        How to combine criteria:
        - 'any': Flat if ANY criterion says flat
        - 'all': Flat only if ALL criteria say flat
        - 'combined': Use weighted combination (recommended)
        - 'cv': Use only coefficient of variation
        - 'entropy': Use only entropy
    
    Returns:
    --------
    is_flat : bool
        True if feature is considered flat
    metrics : dict
        All calculated variability metrics
    """
    metrics = calculate_feature_variability(series)
    
    # Individual flat checks
    cv_flat = metrics['cv'] < cv_threshold
    unique_flat = metrics['unique_ratio'] < unique_threshold
    entropy_flat = metrics['entropy'] < entropy_threshold
    
    if method == 'cv':
        is_flat = cv_flat
    elif method == 'entropy':
        is_flat = entropy_flat
    elif method == 'any':
        is_flat = cv_flat or unique_flat or entropy_flat
    elif method == 'all':
        is_flat = cv_flat and unique_flat and entropy_flat
    elif method == 'combined':
        # Weighted score: higher = more varied
        # Normalize each metric to 0-1 range approximately
        cv_score = min(metrics['cv'] / 0.5, 1.0)  # 0.5 CV = max score
        entropy_score = min(metrics['entropy'] / 4.0, 1.0)  # 4.0 entropy = max score
        unique_score = min(metrics['unique_ratio'] / 0.5, 1.0)  # 50% unique = max score
        
        combined_score = 0.4 * cv_score + 0.4 * entropy_score + 0.2 * unique_score
        is_flat = combined_score < 0.1  # Threshold for combined score
    else:
        raise ValueError(f"Unknown method: {method}")
    
    return is_flat, metrics


def filter_flat_features(
    df: pd.DataFrame,
    feature_columns: Optional[List[str]] = None,
    cv_threshold: float = 0.01,
    unique_threshold: float = 0.01,
    entropy_threshold: float = 1.0,
    method: str = 'combined',
    verbose: bool = True
) -> Tuple[List[str], List[str], pd.DataFrame]:
    """
    Filter out flat/near-flat features from a dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Dataframe containing features
    feature_columns : List[str], optional
        List of feature column names to check.
        If None, uses all numeric columns except 'close', 'date', etc.
    cv_threshold : float
        Coefficient of variation threshold
    unique_threshold : float
        Unique ratio threshold
    entropy_threshold : float
        Entropy threshold
    method : str
        Method to determine flatness
    verbose : bool
        Print details about filtered features
        
    Returns:
    --------
    selected_features : List[str]
        Features that passed the filter (not flat)
    removed_features : List[str]
        Features that were removed (flat)
    metrics_df : pd.DataFrame
        Variability metrics for all features
    """
    # Determine feature columns
    if feature_columns is None:
        exclude_cols = ['close', 'open', 'high', 'low', 'volume', 'date', 
                       'end_date', 'start_date', 'timestamp']
        feature_columns = [col for col in df.select_dtypes(include=[np.number]).columns 
                          if col.lower() not in exclude_cols]
    
    selected_features = []
    removed_features = []
    all_metrics = []
    
    for col in feature_columns:
        is_flat, metrics = is_feature_flat(
            df[col],
            cv_threshold=cv_threshold,
            unique_threshold=unique_threshold,
            entropy_threshold=entropy_threshold,
            method=method
        )
        
        metrics['feature'] = col
        metrics['is_flat'] = is_flat
        all_metrics.append(metrics)
        
        if is_flat:
            removed_features.append(col)
        else:
            selected_features.append(col)
    
    metrics_df = pd.DataFrame(all_metrics)
    metrics_df = metrics_df[['feature', 'is_flat', 'cv', 'entropy', 'std', 
                             'iqr', 'range_ratio', 'unique_ratio', 'mad']]
    
    if verbose:
        print("="*70)
        print("FEATURE VARIABILITY ANALYSIS")
        print("="*70)
        print(f"\nTotal features analyzed: {len(feature_columns)}")
        print(f"Features KEPT (varied): {len(selected_features)}")
        print(f"Features REMOVED (flat): {len(removed_features)}")
        
        if removed_features:
            print(f"\n⚠️  Removed flat features:")
            for feat in removed_features:
                m = metrics_df[metrics_df['feature'] == feat].iloc[0]
                print(f"   - {feat}: CV={m['cv']:.4f}, Entropy={m['entropy']:.2f}")
        
        if selected_features:
            print(f"\n✓ Selected features with variation:")
            for feat in selected_features:
                m = metrics_df[metrics_df['feature'] == feat].iloc[0]
                print(f"   - {feat}: CV={m['cv']:.4f}, Entropy={m['entropy']:.2f}")
        
        print("="*70)
    
    return selected_features, removed_features, metrics_df


def select_varied_features(
    df: pd.DataFrame,
    feature_columns: List[str],
    min_cv: float = 0.01,
    min_entropy: float = 1.0,
    top_n: Optional[int] = None,
    verbose: bool = True
) -> List[str]:
    """
    Simple function to select features with sufficient variation.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Dataframe with features
    feature_columns : List[str]
        Feature column names to evaluate
    min_cv : float
        Minimum coefficient of variation required
    min_entropy : float
        Minimum entropy required
    top_n : int, optional
        If provided, return only top N features by variability
    verbose : bool
        Print summary
        
    Returns:
    --------
    List[str] : Selected feature names
    """
    selected, removed, metrics = filter_flat_features(
        df, 
        feature_columns=feature_columns,
        cv_threshold=min_cv,
        entropy_threshold=min_entropy,
        method='combined',
        verbose=False
    )
    
    if top_n is not None and len(selected) > top_n:
        # Rank by combined variability and take top N
        varied_metrics = metrics[~metrics['is_flat']].copy()
        varied_metrics['variability_score'] = (
            varied_metrics['cv'].rank(pct=True) * 0.4 +
            varied_metrics['entropy'].rank(pct=True) * 0.4 +
            varied_metrics['unique_ratio'].rank(pct=True) * 0.2
        )
        varied_metrics = varied_metrics.sort_values('variability_score', ascending=False)
        selected = varied_metrics.head(top_n)['feature'].tolist()
    
    if verbose:
        print(f"Selected {len(selected)}/{len(feature_columns)} features with sufficient variation")
        if removed:
            print(f"Removed {len(removed)} flat features: {removed}")
    
    return selected


# =============================================================================
# CONVENIENCE FUNCTION FOR YOUR WORKFLOW
# =============================================================================

def filter_tda_features(
    df_features: pd.DataFrame,
    list_features: List[str],
    cv_threshold: float = 0.05,
    entropy_threshold: float = 2.0,
    verbose: bool = True
) -> List[str]:
    """
    Filter TDA features to remove flat/constant ones.
    
    This is the main function to use in your preprocessing pipeline.
    
    Parameters:
    -----------
    df_features : pd.DataFrame
        Feature dataframe (with date as index)
    list_features : List[str]
        List of feature column names
    cv_threshold : float
        Minimum coefficient of variation (default 0.05 = 5%)
        Higher value = stricter filtering
    entropy_threshold : float
        Minimum entropy (default 2.0)
        Higher value = stricter filtering
    verbose : bool
        Print analysis details
    
    Returns:
    --------
    List[str] : Filtered feature names (only varied features)
    
    Example:
    --------
    >>> # Before training
    >>> list_features = get_family_members('TDA_TD_168_SS_720', feature_family)
    >>> list_features = filter_tda_features(df_features, list_features)
    >>> df_features = df_features[list_features]
    """
    selected, removed, metrics = filter_flat_features(
        df_features,
        feature_columns=list_features,
        cv_threshold=cv_threshold,
        entropy_threshold=entropy_threshold,
        method='combined',
        verbose=verbose
    )
    
    if len(selected) == 0:
        warnings.warn("All features were filtered out! Using original features.")
        return list_features
    
    return selected


def filter_tda_features_strict(
    df_features: pd.DataFrame,
    list_features: List[str],
    min_cv: float = 0.1,
    min_range_pct: float = 0.2,
    verbose: bool = True
) -> List[str]:
    """
    Stricter filtering using coefficient of variation and range.
    
    A feature is kept only if:
    - CV (std/mean) > min_cv  (default 10%)
    - Range (max-min)/mean > min_range_pct (default 20%)
    
    Parameters:
    -----------
    df_features : pd.DataFrame
        Feature dataframe
    list_features : List[str]
        Feature column names
    min_cv : float
        Minimum coefficient of variation (0.1 = 10%)
    min_range_pct : float
        Minimum range as % of mean (0.2 = 20%)
    verbose : bool
        Print details
        
    Returns:
    --------
    List[str] : Selected feature names
    """
    selected = []
    removed = []
    
    for col in list_features:
        series = df_features[col].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(series) < 10:
            removed.append((col, 'insufficient data'))
            continue
            
        mean_val = abs(series.mean())
        std_val = series.std()
        range_val = series.max() - series.min()
        
        cv = std_val / mean_val if mean_val > 1e-10 else 0
        range_pct = range_val / mean_val if mean_val > 1e-10 else 0
        
        if cv >= min_cv and range_pct >= min_range_pct:
            selected.append(col)
            if verbose:
                print(f"✓ {col}: CV={cv:.3f}, Range%={range_pct:.3f}")
        else:
            removed.append((col, f'CV={cv:.3f}, Range%={range_pct:.3f}'))
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Selected: {len(selected)}/{len(list_features)} features")
        if removed:
            print(f"\nRemoved {len(removed)} flat features:")
            for feat, reason in removed:
                print(f"  ❌ {feat}: {reason}")
        print(f"{'='*60}")
    
    if len(selected) == 0:
        warnings.warn("All features filtered! Returning original list.")
        return list_features
        
    return selected

In [ ]:
def train_and_evaluate_single_seed(
    df_train: pd.DataFrame,
    df_val: pd.DataFrame,
    list_features: List[str],
    transaction_cost: float,
    seed: int,
    scenario_id: int,
    total_timesteps: int,
    verbose: bool = False
) -> Dict:
    """
    Train and evaluate DQN for a single seed.
    
    Returns dict with all metrics for this seed.
    """
    # Set all seeds
    set_all_seeds(seed)
    
    # Create environments
    def make_env(df, rank):
        def _init():
            set_all_seeds(seed + rank)  # Slightly different seed per env
            return ImprovedTradingEnv(df, transaction_cost=transaction_cost)
        return _init
    
    if USE_SINGLE_ENV:
        n_envs = 1
        train_env = DummyVecEnv([make_env(df_train, 0)])
    else:
        n_envs = 4
        train_env = SubprocVecEnv([make_env(df_train, i) for i in range(n_envs)])
    
    val_env = DummyVecEnv([make_env(df_val, 0)])
    
    # Create and train model
    model = DQN(
        "MlpPolicy",
        train_env,
        learning_rate=1e-4,
        learning_starts=1000,
        buffer_size=50000,
        batch_size=64,
        gamma=0.99,
        target_update_interval=500,
        exploration_fraction=0.3,
        exploration_initial_eps=1.0,
        exploration_final_eps=0.05,
        train_freq=4,
        gradient_steps=1,
        verbose=0,
        seed=seed
    )
    
    model.learn(total_timesteps=total_timesteps)
    
    # Evaluate
    obs = val_env.reset()
    done = False
    
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, done, info = val_env.step(action)
    
    # Extract metrics
    metrics = {
        'seed': seed,
        'total_return': info[0]['total_return'],
        'total_return_before_costs': info[0]['total_return_before_costs'],
        'sharpe_ratio': info[0]['sharpe_ratio'],
        'sharpe_ratio_before_costs': info[0]['sharpe_ratio_before_costs'],
        'sortino_ratio': info[0]['sortino_ratio'],
        'max_drawdown': info[0]['max_drawdown'],
        'win_rate': info[0]['win_rate'],
        'win_rate_before_costs': info[0]['win_rate_before_costs'],
        'total_transaction_costs': info[0]['total_transaction_costs'],
        'trade_count': info[0]['trade_count'],
        'trade_frequency': info[0]['trade_frequency'],
        'final_net_worth': info[0]['net_worth'],
        'final_net_worth_before_costs': info[0]['net_worth_before_costs'],
    }
    
    # Cleanup
    train_env.close()
    val_env.close()
    del model
    
    if verbose:
        print(f"    Seed {seed}: Return={metrics['total_return']*100:.2f}%, Sharpe={metrics['sharpe_ratio']:.3f}")
    
    return metrics

In [ ]:
def aggregate_seed_results(seed_results: List[Dict]) -> Dict:
    """
    Aggregate results from multiple seeds into mean ± std.
    
    Returns dict with:
    - metric_mean: Mean value across seeds
    - metric_std: Standard deviation across seeds
    - metric_min: Minimum value
    - metric_max: Maximum value
    - metric_values: List of all values (for statistical tests)
    """
    df_seeds = pd.DataFrame(seed_results)
    
    aggregated = {
        'n_seeds': len(seed_results),
        'seeds_used': [r['seed'] for r in seed_results],
    }
    
    metrics_to_aggregate = [
        'total_return', 'total_return_before_costs',
        'sharpe_ratio', 'sharpe_ratio_before_costs',
        'sortino_ratio', 'max_drawdown',
        'win_rate', 'win_rate_before_costs',
        'total_transaction_costs', 'trade_count', 'trade_frequency',
        'final_net_worth', 'final_net_worth_before_costs'
    ]
    
    for metric in metrics_to_aggregate:
        values = df_seeds[metric].values
        aggregated[f'{metric}_mean'] = float(np.mean(values))
        aggregated[f'{metric}_std'] = float(np.std(values))
        aggregated[f'{metric}_min'] = float(np.min(values))
        aggregated[f'{metric}_max'] = float(np.max(values))
        aggregated[f'{metric}_values'] = values.tolist()
    
    return aggregated

## Select Scenarios to Process

In [ ]:
# Select scenarios

list_scenario_id = list(range(1, 36)) # All 35 scenarios

total_runs = len(list_scenario_id) * N_SEEDS
print(f"Processing {len(list_scenario_id)} scenarios × {N_SEEDS} seeds = {total_runs} total training runs")
print(f"Estimated time: {total_runs * 2 / 60:.1f} hours (assuming ~2 min per run)")

In [ ]:
# Create results directories
os.makedirs('../RL_outputs/results/df', exist_ok=True)
os.makedirs('../RL_outputs/results/json', exist_ok=True)
os.makedirs('../RL_outputs/results/plot', exist_ok=True)
os.makedirs('../RL_outputs/tensorboard', exist_ok=True)
os.makedirs('../RL_outputs/models', exist_ok=True)
os.makedirs('../RL_outputs/logs', exist_ok=True)
os.makedirs('../RL_outputs/checkpoints', exist_ok=True)

print("Results directories created")

## Main Processing Loop

Process each scenario with improved environment and proper cost tracking

In [ ]:
batch_summary = []
all_seed_results = []  # Store individual seed results for statistical tests

# Track batch results
# (batch_summary and all_seed_results are already defined in the notebook)
for idx, scenario_id in enumerate(list_scenario_id):
    print("\n" + "="*80)
    print(f"Scenario {scenario_id} ({idx+1}/{len(list_scenario_id)})")
    print("="*80)
    
    try:
        # Get scenario data
        scenario_data = get_scenario_data(scenario_id, scenario_config)
        print(f"Asset: {scenario_data['asset']}, Feature Family: {scenario_data['feature_family']}")
        
        transaction_cost = get_transaction_cost(scenario_data['asset'])
        
        # Load and prepare data
        features_full_path = f"../{scenario_data['feature_path']}/{scenario_data['feature_file']}"
        df_features = pd.read_csv(features_full_path)
        
        if scenario_data['feature_family'].startswith('TDA'):
            df_features['date'] = pd.to_datetime(df_features['end_date']).dt.floor('d')
        else:
            df_features['date'] = pd.to_datetime(df_features['date'])
        
        df_features.set_index('date', inplace=True)
        
        list_features = get_family_members(scenario_data['feature_family'], feature_family)
        
        if scenario_data['feature_family'].startswith('TDA'):
            list_features = filter_tda_features_strict(df_features, list_features, verbose=False)
        
        df_features = df_features[list_features]
        
        # Load raw data
        raw_full_path = f"../{scenario_data['raw_path']}/{scenario_data['raw_file']}"
        df_raw = pd.read_csv(raw_full_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw.set_index('date', inplace=True)
        
        # Align features with raw data
        df_features_modified = pd.DataFrame()
        for date in df_features.index:
            if date in df_raw.index:
                df_features_modified = pd.concat([df_features_modified, df_features.loc[[date]]])
            else:
                date_only = date.date()
                df_raw_date = df_raw[df_raw.index.date == date_only]
                if not df_raw_date.empty:
                    last_time = df_raw_date.index.max()
                    df_features_modified = pd.concat([df_features_modified, df_features.loc[[date]].rename(index={date: last_time})])
            df_features_modified = df_features_modified[~df_features_modified.index.duplicated(keep='last')]
        
        df = df_features_modified.join(df_raw, how='inner')
        df = df[list_features + ['close']]
        
        # Split train/val
        df_train = df.loc[(df.index >= scenario_data['start_train_date']) & (df.index <= scenario_data['end_train_date'])].copy()
        df_val = df.loc[(df.index >= scenario_data['start_val_date']) & (df.index <= scenario_data['end_val_date'])].copy()
        
        print(f"Train: {len(df_train)} samples, Val: {len(df_val)} samples")
        
        # Scale features (using training data statistics)
        scalers = {}
        for feature in list_features:
            df_train[feature].replace([np.inf, -np.inf], np.nan, inplace=True)
            df_val[feature].replace([np.inf, -np.inf], np.nan, inplace=True)
            df_train[feature].fillna(method='ffill', inplace=True)
            df_val[feature].fillna(method='ffill', inplace=True)
            
            scaler = MinMaxScaler(feature_range=(0, 1))
            df_train[feature] = scaler.fit_transform(df_train[feature].values.reshape(-1, 1))
            df_val[feature] = scaler.transform(df_val[feature].values.reshape(-1, 1))
            scalers[feature] = scaler
        
        total_timesteps = max(100000, len(df_train) * 50)
        
        # ================================================================
        # RUN MULTIPLE SEEDS (uses train_and_evaluate_single_seed)
        # ================================================================
        print(f"\nRunning {N_SEEDS} seeds...")
        seed_results = []
        
        for seed_idx, seed in enumerate(SEEDS):
            print(f"  Seed {seed_idx+1}/{N_SEEDS} (seed={seed})...", end=" ")
            
            metrics = train_and_evaluate_single_seed(
                df_train=df_train,
                df_val=df_val,
                list_features=list_features,
                transaction_cost=transaction_cost,
                seed=seed,
                scenario_id=scenario_id,
                total_timesteps=total_timesteps,
                verbose=False
            )
            
            # add scenario metadata
            metrics['scenario_id'] = scenario_id
            metrics['asset'] = scenario_data['asset']
            metrics['feature_family'] = scenario_data['feature_family']
            
            seed_results.append(metrics)
            all_seed_results.append(metrics)
            
            print(f"Return={metrics['total_return']*100:.2f}%")
        
        # ================================================================
        # AGGREGATE RESULTS
        # ================================================================
        aggregated = aggregate_seed_results(seed_results)
        aggregated['scenario_id'] = scenario_id
        aggregated['asset'] = scenario_data['asset']
        aggregated['feature_family'] = scenario_data['feature_family']
        aggregated['transaction_cost_pct'] = transaction_cost * 100
        aggregated['n_features'] = len(list_features)
        
        batch_summary.append(aggregated)
        
        # Print summary
        print(f"\n" + "-"*60)
        print(f"RESULTS (n={N_SEEDS} seeds)")
        print("-"*60)
        print(f"Total Return:  {aggregated['total_return_mean']*100:>7.2f}% ± {aggregated['total_return_std']*100:.2f}%")
        print(f"Sharpe Ratio:  {aggregated['sharpe_ratio_mean']:>7.3f} ± {aggregated['sharpe_ratio_std']:.3f}")
        print(f"Win Rate:      {aggregated['win_rate_mean']*100:>7.2f}% ± {aggregated['win_rate_std']*100:.2f}%")
        print(f"Trade Count:   {aggregated['trade_count_mean']:>7.0f} ± {aggregated['trade_count_std']:.0f}")
        print("-"*60)
        
        # Save aggregated results (make JSON-safe)
        save_dir_json = "../RL_outputs/results/json"
        os.makedirs(save_dir_json, exist_ok=True)
        with open(f"{save_dir_json}/{scenario_id}_DQN_results.json", 'w') as f:
            json_safe = {k: v.tolist() if isinstance(v, np.ndarray) else v for k, v in aggregated.items()}
            json.dump(json_safe, f, indent=2)
        
        # Save per-seed results
        save_dir_per_seed = "../RL_outputs/results/json_per_seed"
        os.makedirs(save_dir_per_seed, exist_ok=True)
        pd.DataFrame(seed_results).to_csv(
            f"{save_dir_per_seed}/{scenario_id}_per_seed.csv",
            index=False
        )
        
        print(f"✓ Scenario {scenario_id} complete!")
        
    except Exception as e:
        print(f"\n❌ ERROR in scenario {scenario_id}: {str(e)}")
        traceback.print_exc()
        continue

print("\n" + "="*80)
print("BATCH PROCESSING COMPLETE!")
print("="*80)
print(f"Processed {len(batch_summary)} scenarios successfully")
print(f"Total training runs: {len(all_seed_results)}")


## Batch Results Analysis

In [ ]:
# Create batch summary dataframe
df_batch_summary = pd.DataFrame(batch_summary)
# Save batch summary
df_batch_summary.to_csv('../RL_outputs/results/DQL_batch_summary.csv', index=False)
# Save all seed results (for statistical tests)
df_all_seeds = pd.DataFrame(all_seed_results)
df_all_seeds.to_csv('../RL_outputs/results/DQL_all_seed_results.csv', index=False)